<a href="https://colab.research.google.com/github/Kashaf537/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kashaf537/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — Growing vs. Declining Content

The FlyRank research paper reports that growing pages are structurally
different from declining pages. Growing pages were **37.6% longer** on
average (3.2K vs. 2.3K words) and **20% younger** (184 vs. 230 days old).

The analysis included **74,187 rising pages** and **45,272 falling pages**.

The paper defines trend direction using the change in impressions between
the current 30-day period and the previous 30-day period:

- **Up / Growing:** more than 10% impression growth
- **Down / Declining:** more than 10% impression decline
- **Stable:** within ±10%
- **Flat:** insufficient data
- **New:** created within 30 days

#### My methodology question

How exactly is the growing/declining label constructed, and is the
30-day-vs-previous-30-day impression window applied consistently to every
page included in the comparison?

This matters because the definition of the outcome determines which pages
are considered growing or declining. I would also want to distinguish a
descriptive relationship from a causal claim: the finding shows that
growing and declining pages differ in observed characteristics, but it does
not by itself show that increasing word count or reducing content age will
cause a page to grow.

---

### Finding 2 — Logistic Regression Growth Prediction

The paper reports that a **Logistic Regression model achieved 71% holdout
accuracy** when predicting whether a page was growing or declining.

The model identified **content age** as the strongest negative signal for
growth, while **days visible** and **recent impressions** were among the
strongest positive signals.

The exploratory ML analysis used **61,790 active content records**, filtered
from the larger portfolio to pages with measurable traffic
(`impressions_90d > 0` and `sessions_90d > 0`).

The paper used an **80/20 train/test split** for the Logistic Regression
model.

#### My methodology question

Does the 80/20 random holdout adequately measure how well the model
generalizes to genuinely unseen brands or clients?

The split evaluates unseen content pieces, but the methodology does not
state that entire brands or clients were held out from training. If related
content from the same brand or client can appear in both partitions, the
holdout result may not fully represent performance on a completely unseen
client.

This is a useful limitation to consider because a grouped-by-client split
would provide a stricter test of generalization to clients that were not
represented during training.

In [5]:
# Clone the repository
!git clone https://github.com/Kashaf537/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 211 (delta 87), reused 124 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 2.64 MiB | 9.25 MiB/s, done.
Resolving deltas: 100% (87/87), done.


In [6]:
# Move into the repository
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [7]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    paper	     scripts   submission
CLAUDE.md  docs		notebooks  README.md	     SETUP.md  work
data	   GUIDE.md	outputs    requirements.txt  skills    workflows


In [8]:
from pathlib import Path
import pandas as pd

dataset = Path("data/raw/content_refresh_anonymized.csv")

print("Dataset exists:", dataset.exists())

if dataset.exists():
    df = pd.read_csv(dataset)

    print("Shape:", df.shape)
    print("Columns:", len(df.columns))
    print("\nFirst 5 rows:")
    display(df.head())
else:
    print("Dataset not found!")

Dataset exists: True
Shape: (30000, 44)
Columns: 44

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [9]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [14]:
# Create the target from the trend percentage
target = "is_declining_label"

df["is_declining_label"] = (
    df["trend_pct"] < -10
).astype(int)

print("Target created successfully.")
print(df["is_declining_label"].value_counts())

Target created successfully.
is_declining_label
1    18248
0    11752
Name: count, dtype: int64


In [15]:
model_df = df.dropna(subset=[target]).copy()

X = model_df[feature_columns].copy()
y = model_df[target].astype(int)

# Fill missing numeric values
X = X.fillna(X.median())

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (30000, 28)
y shape: (30000,)

Target distribution:
is_declining_label
1    18248
0    11752
Name: count, dtype: int64


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split

My Week-5 Random Forest achieved a measured Precision@50 of **0.52**
under the original evaluation setup.

To test whether this result generalizes beyond the original split, I
re-ran the model using a **client-grouped train/test split**. This keeps
records from the same client in the same partition, so the test set
contains clients that were not used for training.

This is a stricter evaluation of generalization than a row-level random
split.

In [1]:
from sklearn.model_selection import GroupShuffleSplit

In [16]:
groups = df["client_id"]

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I reviewed the final Random Forest feature set for variables that could
contain information about the outcome that would not be legitimately
available at prediction time.

The main leakage risk is the relationship between the target and the trend
variables. The `is_declining_label` target is derived from the observed
trend direction, so `trend_direction` and `trend_pct` must not be used as
model inputs.

I therefore excluded these variables from the feature set.

In [17]:
forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

used_forbidden = [
    col for col in forbidden_features
    if col in feature_columns
]

print("Forbidden features used:", used_forbidden)

Forbidden features used: []


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original Week-5 claim

> The Random Forest improves Precision@50 from 0.36 to 0.52.

### Revised claim

> Under the original evaluation setup, the Random Forest achieved a
> measured Precision@50 of **0.52**, compared with **0.36** for the Week-4
> baseline. This is a **16 percentage-point observed improvement** in this
> dataset and evaluation.

The client-grouped evaluation provides a stricter test of whether this
result generalizes to unseen clients. Therefore, the model should be
treated as **decision-support for prioritization**, rather than evidence
that the Random Forest will always outperform the baseline in production.

The results are directional and measured on this dataset; they do not
establish that the model will produce the same performance on new clients
or future data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.